# C?ng ?o?n 02 ? Chia d? li?u Train/Test
## M?c ti?u v? ph??ng ph?p
M?c ti?u 80/20, gi? 33 ng?nh, ng?n r? r? vector F54 tr?ng. T?i hi?n l?m s?ch ?? x?c nh?n ? Notebook 01 b?ng c?c cell ??c l?p d??i ??y: gi? Likert v?n b?n, trim nh?n, kh?ng x?a m?u. Gi? th? t? q1?q54 c?a workbook. D?ng StratifiedGroupKFold 5 folds, shuffle=True, random_state=42, l?y fold ??u ti?n c? ??nh; kh?ng t?i ?u fold/seed. Group ch? d?a v?o vector X. Kh?ng chu?n h?a, ch?n ??c tr?ng ho?c hu?n luy?n.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

assert Path(sys.executable).resolve() == Path(r'C:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe').resolve()
print(f'Kernel sys.executable: {sys.executable}')

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'data_AI_1000_rows.xlsx').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy workbook từ thư mục hiện tại hoặc thư mục cha.')

workbook_path = ROOT / 'data_AI_1000_rows.xlsx'
sheet_name = 'AUGMENTED_RAW_1000'
excel_file = pd.ExcelFile(workbook_path)
assert excel_file.sheet_names == [sheet_name]

df_original = pd.read_excel(workbook_path, sheet_name=sheet_name)
metadata_columns = ['Dấu thời gian', 'Tên Sinh viên', 'Ngành học', 'Giới tính']
label_column = 'Ngành học'
question_columns = list(df_original.columns[4:])

assert df_original.shape == (1000, 58)
assert list(df_original.columns[:4]) == metadata_columns
assert len(question_columns) == 54
assert label_column not in question_columns

print(f'Worksheet: {sheet_name}')
print(f'Dữ liệu gốc: {len(df_original)} dòng × {len(df_original.columns)} cột')
print(f'Số câu hỏi: {len(question_columns)}; cột nhãn: {label_column}')

Kernel sys.executable: C:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe
Worksheet: AUGMENTED_RAW_1000
Dữ liệu gốc: 1000 dòng × 58 cột
Số câu hỏi: 54; cột nhãn: Ngành học


In [2]:
answer_type_rows = []
for column in question_columns:
    value_types = sorted({type(value).__name__ for value in df_original[column].dropna()})
    answer_type_rows.append({
        'Câu hỏi': column,
        'dtype pandas': str(df_original[column].dtype),
        'Kiểu giá trị': ', '.join(value_types),
        'Ô thiếu': int(df_original[column].isna().sum()),
        'Giá trị phân biệt': int(df_original[column].nunique(dropna=True)),
    })
answer_type_table = pd.DataFrame(answer_type_rows)

expected_answer_labels = [
    'Chưa bao giờ đúng',
    'Chỉ đúng 1 vài trường hợp',
    'Chỉ đúng 1 nửa',
    'Gần như là đúng',
    'Hoàn toàn đúng',
]
answer_cells = df_original[question_columns]
missing_answers = int(answer_cells.isna().sum().sum())
unknown_answers = int((answer_cells.notna() & ~answer_cells.isin(expected_answer_labels)).sum().sum())
numeric_answers = answer_cells.apply(pd.to_numeric, errors='coerce')
all_answers_are_numeric = bool(numeric_answers.notna().all().all())
numeric_values_are_valid = bool(
    all_answers_are_numeric
    and numeric_answers.ge(1).all().all()
    and numeric_answers.le(5).all().all()
    and numeric_answers.mod(1).eq(0).all().all()
)

display(answer_type_table)
print(f'Ô thiếu trong X: {missing_answers}')
print(f'Câu trả lời ngoài 5 nhãn Likert đã xác nhận: {unknown_answers}')
print(f'Tất cả câu trả lời đã là số nguyên 1–5: {numeric_values_are_valid}')

                                              Câu hỏi  ... Giá trị phân biệt
0                               1. Tôi có tính tự lập  ...                 5
1                             2. Tôi suy nghĩ thực tế  ...                 5
2       3. Tôi là người thích nghi với môi trường mới  ...                 5
3   4. Tôi có thể vận hành, điều khiển các máy móc...  ...                 5
4   5. Tôi làm các công việc thủ công như gấp giấy...  ...                 5
5   6. Tôi thích tiếp xúc với thiên nhiên động vật...  ...                 5
6   7. Tôi thích những công việc sử dụng tay chân ...  ...                 5
7     8. Tôi thích những công việc  thấy ngay kết quả  ...                 5
8   9. Tôi thích những công việc ngoài trời hơn là...  ...                 5
9       10. Tôi có tìm hiểu khám phá nhiều vấn đề mới  ...                 5
10               11. Tôi có khả năng phân tích vấn đề  ...                 5
11  12. Tôi biết suy nghĩ một cách mạch lạc, chặt chẽ  ...                 5

In [3]:
y_original = df_original[label_column].astype('string')
y_trimmed = y_original.str.strip()
trimmed_label_rows = int((y_original != y_trimmed).fillna(False).sum())

label_forms = pd.DataFrame({'Tên gốc': y_original, 'Tên sau trim': y_trimmed})
label_forms = label_forms.drop_duplicates()
label_forms['Khóa không phân biệt hoa thường'] = label_forms['Tên sau trim'].str.casefold()
label_variants = (
    label_forms.groupby('Khóa không phân biệt hoa thường', dropna=False)
    .agg(
        Số_cách_viết=('Tên sau trim', 'nunique'),
        Các_tên=('Tên sau trim', lambda values: ' | '.join(sorted(values.dropna().unique()))),
    )
    .reset_index(drop=True)
)
label_variants = label_variants[label_variants['Số_cách_viết'] > 1]

label_count_before = y_original.value_counts(dropna=False).rename('Trước làm sạch')
label_count_after = y_trimmed.value_counts(dropna=False).rename('Sau làm sạch')
label_count_comparison = pd.concat([label_count_before, label_count_after], axis=1).fillna(0).astype(int)
label_count_comparison['Chênh lệch'] = label_count_comparison['Sau làm sạch'] - label_count_comparison['Trước làm sạch']
label_count_comparison = label_count_comparison.sort_values('Sau làm sạch', ascending=False, kind='stable')

print(f'Số hàng có khoảng trắng đầu/cuối được loại bỏ: {trimmed_label_rows}')
print(f'Số nhãn phân biệt trước/sau: {y_original.nunique(dropna=True)}/{y_trimmed.nunique(dropna=True)}')
print('Các nhóm tên khác nhau chỉ bởi chữ hoa/thường sau khi trim:')
display(label_variants)
display(label_count_comparison)

Số hàng có khoảng trắng đầu/cuối được loại bỏ: 0
Số nhãn phân biệt trước/sau: 33/33
Các nhóm tên khác nhau chỉ bởi chữ hoa/thường sau khi trim:
Empty DataFrame
Columns: [Số_cách_viết, Các_tên]
Index: []
                                    Trước làm sạch  Sau làm sạch  Chênh lệch
Ngành học                                                                   
Công nghệ thông tin                             73            73           0
Giáo dục tiểu học                               46            46           0
Ngôn ngữ anh                                    44            44           0
Luật                                            40            40           0
Kế toán                                         29            29           0
Giáo dục chính trị                              28            28           0
Kỹ thuật phần mềm                               28            28           0
Sư phạm ngữ văn                                 28            28           0
Quản lý đất đai            

In [4]:
duplicate_answer_mask = answer_cells.duplicated(keep=False)
duplicate_rows = df_original.loc[duplicate_answer_mask, question_columns + [label_column]].copy()
duplicate_groups = (
    duplicate_rows.groupby(question_columns, dropna=False)
    .agg(
        Số_bản_ghi=(label_column, 'size'),
        Số_nhãn_ngành=(label_column, 'nunique'),
        Các_nhãn_ngành=(label_column, lambda values: ' | '.join(sorted(values.astype(str).unique()))),
    )
    .reset_index(drop=True)
)
conflicting_duplicate_groups = duplicate_groups[duplicate_groups['Số_nhãn_ngành'] > 1]

print(f'Số hàng thuộc nhóm có vector trả lời lặp: {int(duplicate_answer_mask.sum())}')
print(f'Số nhóm vector lặp: {len(duplicate_groups)}')
print(f'Số nhóm vector lặp có nhãn ngành mâu thuẫn: {len(conflicting_duplicate_groups)}')
display(duplicate_groups)

if not conflicting_duplicate_groups.empty:
    print('DỪNG: X trùng nhưng y khác. Không tạo dữ liệu sạch; cần chủ dự án quyết định cách xử lý.')
else:
    print('Không phát hiện mâu thuẫn X/y trong các nhóm vector trả lời lặp.')

Số hàng thuộc nhóm có vector trả lời lặp: 4
Số nhóm vector lặp: 2
Số nhóm vector lặp có nhãn ngành mâu thuẫn: 0
   Số_bản_ghi  Số_nhãn_ngành       Các_nhãn_ngành
0           2              1   Quản trị khách sạn
1           2              1  Công nghệ thông tin
Không phát hiện mâu thuẫn X/y trong các nhóm vector trả lời lặp.


In [5]:
if not conflicting_duplicate_groups.empty:
    raise RuntimeError('Có nhóm X trùng nhưng y khác; dừng trước khi tạo dữ liệu sạch.')
if missing_answers or unknown_answers:
    raise RuntimeError('X có giá trị thiếu hoặc câu trả lời ngoài miền đã xác nhận; cần quyết định trước khi làm sạch.')
if y_trimmed.isna().any() or y_trimmed.eq('').any():
    raise RuntimeError('Nhãn ngành bị thiếu hoặc rỗng sau khi trim; không tự điền nhãn.')

if numeric_values_are_valid:
    cleaned_answers = numeric_answers.astype('int64')
    answer_handling = 'Đã chuyển câu trả lời số nguyên 1–5 sang int64.'
else:
    cleaned_answers = answer_cells.copy()
    answer_handling = 'Giữ nguyên 5 nhãn Likert dạng văn bản; không cần ép kiểu số.'

df_clean = cleaned_answers.copy()
df_clean[label_column] = y_trimmed
X = df_clean[question_columns].copy()
y = df_clean[label_column].copy()

assert len(df_clean) == len(df_original)
assert X.shape == (len(df_original), 54)
assert list(X.columns) == question_columns
assert label_column not in X.columns
assert not set(metadata_columns).intersection(X.columns)
assert len(y) == len(X)

before_after_summary = pd.DataFrame([
    {'Chỉ số': 'Số bản ghi', 'Trước': len(df_original), 'Sau': len(df_clean)},
    {'Chỉ số': 'Số đặc trưng X', 'Trước': len(question_columns), 'Sau': X.shape[1]},
    {'Chỉ số': 'Số nhãn ngành y', 'Trước': y_original.nunique(), 'Sau': y.nunique()},
    {'Chỉ số': 'Số hàng có nhãn được trim', 'Trước': 0, 'Sau': trimmed_label_rows},
    {'Chỉ số': 'Số hàng bị xóa', 'Trước': 0, 'Sau': len(df_original) - len(df_clean)},
], columns=['Chỉ số', 'Trước', 'Sau'])

print(answer_handling)
display(before_after_summary)
display(label_count_comparison)

Giữ nguyên 5 nhãn Likert dạng văn bản; không cần ép kiểu số.
                      Chỉ số  Trước   Sau
0                 Số bản ghi   1000  1000
1             Số đặc trưng X     54    54
2            Số nhãn ngành y     33    33
3  Số hàng có nhãn được trim      0     0
4             Số hàng bị xóa      0     0
                                    Trước làm sạch  Sau làm sạch  Chênh lệch
Ngành học                                                                   
Công nghệ thông tin                             73            73           0
Giáo dục tiểu học                               46            46           0
Ngôn ngữ anh                                    44            44           0
Luật                                            40            40           0
Kế toán                                         29            29           0
Giáo dục chính trị                              28            28           0
Kỹ thuật phần mềm                               28            28       

## Ph?n chia v? ph?n b? ng?nh
H? tr? t?i thi?u: m?i ng?nh c? ?t nh?t m?t m?u trong m?i t?p. Thi?u h? tr? th? b?o c?o v? d?ng ?? xem x?t, kh?ng x?a/g?p ng?nh.

In [6]:
from sklearn.model_selection import StratifiedGroupKFold
assert y.nunique() == 33
groups = pd.Series(pd.factorize(pd.Series(list(X.itertuples(index=False, name=None))), sort=False)[0], index=X.index)
sizes = groups.value_counts()
assert (sizes > 1).sum() == 2 and sizes[sizes > 1].sum() == 4
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_pos, test_pos = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_pos].copy(), X.iloc[test_pos].copy()
y_train, y_test = y.iloc[train_pos].copy(), y.iloc[test_pos].copy()
summary = pd.DataFrame({'S? m?u':[len(X_train),len(X_test)], 'T? l? (%)':[len(X_train)/len(X)*100,len(X_test)/len(X)*100], 'S? ng?nh':[y_train.nunique(),y_test.nunique()]}, index=['Train','Test'])
display(summary)
counts = pd.DataFrame({'T?ng':y.value_counts(),'Train':y_train.value_counts(),'Test':y_test.value_counts()}).fillna(0).astype(int).sort_index()
for name, total in [('T?ng',len(y)),('Train',len(y_train)),('Test',len(y_test))]:
    counts[name+' (%)'] = counts[name]/total*100
counts['Test - Train (?i?m %)'] = counts['Test (%)']-counts['Train (%)']
counts['Test - T?ng (?i?m %)'] = counts['Test (%)']-counts['T?ng (%)']
display(counts.round(3))
print('Ch?nh l?ch tuy?t ??i t?i ?a Test/Train:',counts['Test - Train (?i?m %)'].abs().max())
print('Ch?nh l?ch tuy?t ??i t?i ?a Test/T?ng:',counts['Test - T?ng (?i?m %)'].abs().max())
if (counts[['Train','Test']] < 1).any().any():
    raise RuntimeError('D?NG: thi?u h? tr? cho 33 ng?nh. C?n xem x?t c?ch chia.')

       S? m?u  T? l? (%)  S? ng?nh
Train     800       80.0        33
Test      200       20.0        33
                                    T?ng  ...  Test - T?ng (?i?m %)
Ngành học                                 ...                      
Công nghệ kỹ thuật hóa học            28  ...                   0.2
Công nghệ kỹ thuật xây dựng           28  ...                  -0.3
Công nghệ thông tin                   73  ...                   0.2
Công tác xã hội                       27  ...                  -0.2
Giáo dục chính trị                    28  ...                  -0.3
Giáo dục mầm non                      27  ...                  -0.2
Giáo dục tiểu học                     46  ...                  -0.1
Kinh tế                               28  ...                   0.2
Kế toán                               29  ...                   0.1
Kỹ thuật phần mềm                     28  ...                  -0.3
Kỹ thuật điện                         27  ...                  -0.2
Kỹ thuật đi

## Ki?m tra leakage
Index v? group kh?ng l? ??c tr?ng. Code ch? ch?n c?t, ki?m tra mi?n v? trim chu?i tr??c split; kh?ng g?i b?t k? ph?p fit n?o.

In [7]:
assert len(X_train)+len(X_test) == len(X)
assert X.index.is_unique and X_train.index.intersection(X_test.index).empty
assert set(X_train.index) | set(X_test.index) == set(X.index)
assert set(groups.iloc[train_pos]).isdisjoint(groups.iloc[test_pos])
assert set(X_train.itertuples(index=False,name=None)).isdisjoint(X_test.itertuples(index=False,name=None))
assert X_train.shape[1] == X_test.shape[1] == 54
assert list(X_train.columns) == list(X_test.columns) == question_columns
assert not set(metadata_columns).intersection(X.columns)
assert y.name == label_column
assert y.equals(df_original[label_column].astype('string').str.strip())
assert y_train.equals(y.loc[X_train.index]) and y_test.equals(y.loc[X_test.index])
assert set(y_train) == set(y_test) == set(y)
assert X_train.equals(df_original.loc[X_train.index,question_columns])
assert X_test.equals(df_original.loc[X_test.index,question_columns])
report = sizes[sizes > 1].rename('S? m?u').to_frame()
report['T?p'] = ['Train' if g in set(groups.iloc[train_pos]) else 'Test' for g in report.index]
display(report)
print('PASS: to?n b? ki?m tra leakage; 2 nh?m/4 m?u tr?ng n?m tr?n t?ng t?p.')
print('H? tr? nh? nh?t Train/Test:',counts['Train'].min(),counts['Test'].min())

     S? m?u    T?p
392       2  Train
394       2   Test
PASS: to?n b? ki?m tra leakage; 2 nh?m/4 m?u tr?ng n?m tr?n t?ng t?p.
H? tr? nh? nh?t Train/Test: 21 5


## Nh?n x?t v? k?t lu?n
B?ng tr?nh b?y ?? ph?n b? 33 ng?nh; kh?ng c?n bi?u ?? b? sung. T? l? ph?n t?ng ch? x?p x? do r?ng bu?c nh?m. ?? l?p ch?a b?o ??m ?? m?u ?? ??nh gi? tin c?y; ch?a ??nh gi? kh? n?ng t?ng qu?t ngo?i workbook.

Gi? c?ng th? t? d? li?u, nh?m vector, 5 folds, shuffle=True, random_state=42 v? fold ??u ti?n trong c?c notebook ti?p theo. Ch? l?m s?ch kh?ng h?c tham s? tr??c split. StandardScaler, MinMaxScaler, feature selection c? h?c tham s?, PCA v? hyperparameter tuning ph?i fit tr?n Train ho?c trong training folds, kh?ng fit Test. C?c t?p ch? l?u trong b? nh?. D?ng t?i Notebook 02.